In [1]:
import os
import re
import glob
import torch
import tempfile
import numpy as np
from datasets import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders, normalizers
from transformers import (
    AutoTokenizer, PreTrainedTokenizerFast,
    LlamaConfig, LlamaForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    TrainerCallback
)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Загрузка и препроцессинг данных

In [3]:
def load_and_preprocess(data_dir="data"):
    print("Загрузка текстов...")
    texts = []
    for fpath in glob.glob(os.path.join(data_dir, "**/*.txt"), recursive=True):
        with open(fpath, "r", encoding="utf-8") as f:
            texts.append(f.read())
            
    raw_text = "\n".join(texts)
    lines = raw_text.split("\n")
    
    print("🧹 Очистка и фильтрация...")
    latin_re = re.compile(r'[A-Za-z]')
    cleaned = []
    
    for line in lines:
        line = line.strip()
        if not line: continue
            
        # Удаляем строки с латинскими буквами
        if latin_re.search(line): continue
            
        # Нормализация повторяющейся пунктуации
        line = re.sub(r'([.,!?;:—\-])\1+', r'\1', line)
        line = re.sub(r'\.{3,}', '…', line)
        line = re.sub(r'\s+', ' ', line).strip()
        
        if line:
            cleaned.append(line)
            
    # Удаление дубликатов с сохранением порядка
    seen = set()
    unique_cleaned = []
    for line in cleaned:
        if line not in seen:
            seen.add(line)
            unique_cleaned.append(line)
            
    print(f"Очищено строк: {len(unique_cleaned)}")
    return unique_cleaned

cleaned_lines = load_and_preprocess("data")

Загрузка текстов...
🧹 Очистка и фильтрация...
Очищено строк: 295374


## 2. Чанкирование для подготовки к токенизации

In [4]:
def chunk_texts(lines, chunk_char_size=1500):
    chunks = []
    current_chunk = []
    current_len = 0
    
    for line in lines:
        if current_len + len(line) + 1 > chunk_char_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = [line]
            current_len = len(line)
        else:
            current_chunk.append(line)
            current_len += len(line) + 1  # +1 for space
            
    if current_chunk:
        chunks.append(" ".join(current_chunk))
        
    print(f"Создано чанков: {len(chunks)}")
    return chunks

chunks = chunk_texts(cleaned_lines)

Создано чанков: 28543


In [5]:
chunks[:2]

['КАПИТАНСКАЯ ДОЧКА Береги честь смолоду. Пословица. СЕРЖАНТ ГВАРДИИ - Был бы гвардии он завтра ж капитан. - Того не надобно; пусть в армии послужит. - Изрядно сказано! пускай его потужит. . . . . . . . . . . . . . . . Да кто его отец? Княжнин. Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17. году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве.',
 'Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С пятилетнего возраста отдан я был на руки с

## 3. Обучение токенизатора

In [6]:
def train_custom_tokenizer(text_chunks, vocab_size=3000, save_path="custom_tokenizer"):
    print("Обучение токенизатора...")
    tokenizer = Tokenizer(models.BPE())
    tokenizer.normalizer = None
    
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
    tokenizer.decoder = decoders.ByteLevel()
    
    special_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"]
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=special_tokens)
    
    with tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".txt", encoding="utf-8") as f:
        f.write("\n".join(text_chunks))
        f.flush()
        tokenizer.train([f.name], trainer)
        os.remove(f.name)
        
    os.makedirs(save_path, exist_ok=True)
    tokenizer.save(os.path.join(save_path, "tokenizer.json"))
    
    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        bos_token="<bos>", eos_token="<eos>", pad_token="<pad>", unk_token="<unk>"
    )
    hf_tokenizer.save_pretrained(save_path)
    print(f"Токенизатор сохранён в {save_path}")
    return hf_tokenizer

tokenizer = train_custom_tokenizer(chunks, vocab_size=3000, save_path="custom_tokenizer")

Обучение токенизатора...



Токенизатор сохранён в custom_tokenizer


In [7]:
test_text = "Все мысли, которые имеют огромные последствия, требуют внимания."
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens, skip_special_tokens=True)

print("Оригинал :", test_text)
print("Токены    :", tokens)
print("Декодирово:", decoded)
assert decoded.strip() == test_text.strip(), "Декодер работает некорректно!"

Оригинал : Все мысли, которые имеют огромные последствия, требуют внимания.
Токены    : [1036, 1752, 14, 1004, 463, 121, 807, 2275, 377, 1007, 2444, 14, 2117, 2569, 1444, 901, 16]
Декодирово:  Все мысли, которые имеют огромные последствия, требуют внимания.


## 4. Подготовка датасета

In [17]:
def prepare_dataset(chunks, tokenizer, max_length=512):
    print("Токенизация и форматирование датасета...")
    dataset = Dataset.from_dict({"text": chunks})
    
    def tokenize_fn(examples):
        encodings = tokenizer(
            examples["text"], 
            truncation=True, 
            max_length=max_length,
            add_special_tokens=True,
            padding=False
        )
        return encodings
        
    tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) >= 10)
    print(f"Итоговый размер датасета: {len(tokenized_ds)} примеров")
    return tokenized_ds

train_dataset = prepare_dataset(chunks, tokenizer, max_length=512)

Токенизация и форматирование датасета...


Map:   0%|          | 0/28543 [00:00<?, ? examples/s]

Filter:   0%|          | 0/28543 [00:00<?, ? examples/s]

Итоговый размер датасета: 28536 примеров


In [18]:
train_dataset[0]

{'input_ids': [271,
  1512,
  1469,
  2434,
  1859,
  1512,
  1167,
  1717,
  1726,
  1512,
  2381,
  298,
  1812,
  2540,
  1726,
  1512,
  392,
  158,
  360,
  123,
  167,
  1104,
  394,
  962,
  129,
  16,
  1380,
  805,
  2456,
  16,
  257,
  76,
  105,
  76,
  116,
  76,
  106,
  1512,
  1167,
  1859,
  382,
  1386,
  1512,
  76,
  116,
  1783,
  2434,
  2434,
  161,
  392,
  683,
  205,
  192,
  900,
  142,
  618,
  237,
  2126,
  216,
  1723,
  207,
  206,
  16,
  161,
  311,
  222,
  175,
  444,
  1447,
  29,
  136,
  2081,
  138,
  2864,
  123,
  525,
  248,
  207,
  16,
  161,
  1264,
  1896,
  165,
  322,
  1498,
  4,
  136,
  1035,
  336,
  268,
  432,
  248,
  207,
  16,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  1806,
  558,
  908,
  268,
  1522,
  33,
  1879,
  236,
  2238,
  16,
  253,
  428,
  194,
  963,
  1704,
  2573,
  382,
  127,
  209,
  227,
  138,
  848,
  769,
  902,
  1411,
  166,
  293,
 

## 5. Инициализация модели

In [25]:
import gc

# Удаление модели и освобождение памяти
# del llm
gc.collect()

# Дополнительная очистка GPU памяти (если используется CUDA)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [26]:
print("Инициализация модели Llama...")
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=len(tokenizer),
    max_position_embeddings=512,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    use_cache=True
)

model = LlamaForCausalLM(config).to(device)
print(f"Количество параметров: {model.num_parameters():,}")

Инициализация модели Llama...
Количество параметров: 132,006,912


## 6. Коллбэк для валидации на промптах

In [16]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

class GenerationEvalCallback(TrainerCallback):
    def __init__(self, prompts, tokenizer, device, log_every_steps=200):
        self.prompts = prompts
        self.tokenizer = tokenizer
        self.device = device
        self.log_every = log_every_steps
        
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.log_every == 0:
            self._generate(state.global_step, kwargs["model"])
            
    def _generate(self, step, model):
        model.eval()
        print(f"\n{'='*20} EVAL STEP {step} {'='*20}")
        for prompt in self.prompts[:4]:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, 
                    max_new_tokens=40, 
                    do_sample=True, 
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.1
                )
            gen_text = self.tokenizer.decode(out[0], skip_special_tokens=True)
            print(f"{prompt}\n {gen_text}\n")
        model.train()
        print(f"{'='*50}\n")

## 7. Настройка Trainer и запуск обучения

In [ ]:
training_args = TrainingArguments(
    output_dir="./rus_lit_llama_pretrain",
    per_device_train_batch_size=24,
    gradient_accumulation_steps=4,  # 24 * 4 = 96 эффективный batch_size
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    num_train_epochs=3,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="no",
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    callbacks=[GenerationEvalCallback(test_prompts, tokenizer, device, log_every_steps=200)]
)

print("Запуск обучения...")
trainer.train()

Запуск обучения...


Step,Training Loss
50,6.936187
100,6.406120
150,5.907480
200,5.549887
250,5.262088
300,5.064704
350,4.891594



==================== EVAL STEP 200 ====================
Все мысли, которые имеют огромные последствия
  Все мысли, которые имеют огромные последствия. Я с ними в тене и на него ее, что это, что он не только что это, что я не мог бы и, и я так же он не не не так#

Сила войска зависит от его духа
  Сила войска зависит от его духа в рерные и разая, что она, когда не села. Встя, и все выразится к нему. - Я, - и, не было не забыл,

Мысль о том, что он принес страдания
  Мысль о том, что он принес страдания. Вы не мог бы и не знаю, что он, когда я не только так как будто она не только и не с темом. Я бы, что я не любить. Я это не

Человек сознает себя свободным
  Человек сознает себя свободнымно не него, в бахром и мела на лале. Она калерил его на меня на Бавей. На что он в коле? А я к




Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 8. Финальная генерация на тестовых промптах

In [ ]:
model.eval()
model.to(device)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=60, 
            do_sample=True, 
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Промпт: {prompt}")
    print(f"Ответ:  {generated}")
    print("-"*60)